# 2D Hubbard Ground-State Energy with DMRG

This notebook only estimates the ground-state energy of the 2D Hubbard model on a small lattice.

In [1]:
using ITensors
using ITensorMPS

# ------------------- User parameters -------------------
const Lx = 4
const Ly = 4
const N = Lx * Ly

const t = 1.0
const U = 2.0
const pbc = true
const conserve_qns = true

# ------------------- 2D to 1D snake mapping -------------------
@inline function site_index(x::Int, y::Int)
    if isodd(y)
        return (y - 1) * Lx + x
    else
        return (y - 1) * Lx + (Lx - x + 1)
    end
end

function build_bonds(Lx::Int, Ly::Int; pbc::Bool=false)
    bonds = Vector{NTuple{2, Int}}()

    for y in 1:Ly
        for x in 1:(Lx - 1)
            push!(bonds, (site_index(x, y), site_index(x + 1, y)))
        end
        if pbc
            push!(bonds, (site_index(Lx, y), site_index(1, y)))
        end
    end

    for x in 1:Lx
        for y in 1:(Ly - 1)
            push!(bonds, (site_index(x, y), site_index(x, y + 1)))
        end
        if pbc
            push!(bonds, (site_index(x, Ly), site_index(x, 1)))
        end
    end

    bonds = map(b -> (min(b[1], b[2]), max(b[1], b[2])), bonds)
    sort!(bonds)
    unique!(bonds)
    return bonds
end

bonds = build_bonds(Lx, Ly; pbc=pbc)
sites = siteinds("Electron", N; conserve_qns=conserve_qns)

# ------------------- Hamiltonian -------------------
os = OpSum()
for (i, j) in bonds
    os += -t, "Cdagup", i, "Cup", j
    os += -t, "Cdagup", j, "Cup", i
    os += -t, "Cdagdn", i, "Cdn", j
    os += -t, "Cdagdn", j, "Cdn", i
end
for i in 1:N
    os += U, "Nup", i, "Ndn", i
end
H = MPO(os, sites)

# ------------------- Initial state -------------------
state = Vector{String}(undef, N)
for i in 1:N
    state[i] = isodd(i) ? "Up" : "Dn"
end
psi0 = MPS(sites, state)

# ------------------- DMRG parameters -------------------
nsweeps = 10
maxdim = [500]
cutoff = [1e-10]
noise = [1e-6, 1e-7, 1e-8, 0.0]

# ------------------- Ground-state energy -------------------
energy, psi = dmrg(H, psi0; nsweeps, maxdim, cutoff, noise)

println("------ Results ------")
println("Model: 2D Hubbard")
println("Lx x Ly = $(Lx) x $(Ly), N = $N, PBC = $pbc")
println("Parameters: t = $t, U = $U, mu = $mu")
println("Ground-state energy E0 = $energy")
println("Energy per site e0 = $(energy / N)")


After sweep 1 energy=-15.248515119927697  maxlinkdim=500 maxerr=2.15E-08 time=27.700
After sweep 2 energy=-15.450839581408355  maxlinkdim=500 maxerr=5.33E-04 time=11.960
After sweep 3 energy=-15.466149640916761  maxlinkdim=500 maxerr=7.47E-04 time=12.177
After sweep 4 energy=-15.469526253260542  maxlinkdim=500 maxerr=8.22E-04 time=9.777
After sweep 5 energy=-15.470883790718938  maxlinkdim=500 maxerr=8.91E-04 time=9.022
After sweep 6 energy=-15.471534222316306  maxlinkdim=500 maxerr=9.33E-04 time=9.126
After sweep 7 energy=-15.472088365513537  maxlinkdim=500 maxerr=9.68E-04 time=8.981
After sweep 8 energy=-15.472528147737188  maxlinkdim=500 maxerr=9.96E-04 time=9.023
After sweep 9 energy=-15.47263227364914  maxlinkdim=500 maxerr=1.02E-03 time=8.188
After sweep 10 energy=-15.47262528301605  maxlinkdim=500 maxerr=1.04E-03 time=7.782
------ Results ------
Model: 2D Hubbard
Lx x Ly = 4 x 4, N = 16, PBC = false
Parameters: t = 1.0, U = 2.0, mu = 0.0
Ground-state energy E0 = -15.4726252830160